# The Two-Period Consumption-Savings Problem — Computational Lab

**Course**: Constant Elasticity Dynamics  
**Instructor**: Saki Bigio, UCLA

---

This notebook generates figures for the section on the two-period consumption-savings problem. Three features distinguish this from the static CES problem:

1. **The role of the IES $\theta$.** A change in $R$ alters the intertemporal price; the IES governs how strongly the agent substitutes. Cobb-Douglas ($\theta = 1$) is the knife-edge where the compensated substitution effect on $c_0$ vanishes.

2. **Wealth effects from rate changes.** Lifetime wealth $W = b_0 + y_0 + y_1/R$ depends on $R$ itself. Changes in $R$ therefore re-value the present value of future income — a channel absent from the static problem.

3. **Savers vs. borrowers.** The distributional consequences of a rate change flip sign with the savings position: savers with $a > 0$ gain, borrowers with $a < 0$ lose. The welfare consequences are qualitatively opposite even though both face the same price change.

---

**Sections**

1. Setup and the canonical mapping  
2. The Euler equation and consumption growth  
3. Budget set, indifference curves, and the optimum  
4. The role of the IES $\theta$  
5. Wealth effects from interest rate changes  
6. **Geometric Slutsky decomposition: budget rotation and the saver/borrower split**  
7. The algebraic Hicksian Slutsky decomposition  
8. Savers vs. borrowers: the price + savings view  
9. Welfare: who gains, who loses?  
10. Exercises

## Setup

In [ ]:
using Plots, LinearAlgebra, Printf, LaTeXStrings

# ----- Plot defaults -----
default(
    fontfamily = "Computer Modern",
    framestyle = :box,
    grid = true,
    gridalpha = 0.25,
    linewidth = 2,
    legend = :best,
    size = (700, 450),
    margin = 5Plots.mm,
)

# Colors — consistent with earlier notebooks
C_BLUE   = RGB(0.16, 0.37, 0.66)
C_RED    = RGB(0.78, 0.22, 0.22)
C_GREEN  = RGB(0.20, 0.55, 0.30)
C_ORANGE = RGB(0.90, 0.50, 0.10)
C_PURPLE = RGB(0.45, 0.25, 0.55)
C_GREY   = RGB(0.45, 0.45, 0.45)

# ----- Core model functions -----
"""Period utility for CRRA with IES θ (log when θ ≈ 1)."""
u(c, θ) = abs(θ - 1.0) < 1e-8 ? log(c) : (c^(1 - 1/θ) - 1) / (1 - 1/θ)

"""Lifetime wealth in present value terms."""
W_life(b0, y0, y1, R) = b0 + y0 + y1/R

"""Canonical aggregator weights from β and θ."""
function canonical_weights(β, θ)
    ω1 = β^θ / (1 + β^θ)
    ω0 = 1.0 - ω1
    return ω0, ω1
end

"""Solve the two-period problem. Returns (c0, c1, W, a, w0, w1)."""
function solve_two_period(; b0=0.0, y0=1.0, y1=1.0, β=0.96, θ=1.0, R=1.04)
    W      = W_life(b0, y0, y1, R)
    ω0, ω1 = canonical_weights(β, θ)
    denom  = ω0 + ω1 * R^(θ - 1)
    w0     = ω0 / denom                 # expenditure share on c0
    w1     = ω1 * R^(θ - 1) / denom     # expenditure share on q·c1
    c0     = w0 * W
    c1     = w1 * R * W
    a      = y0 + b0 - c0               # savings (positive => saver)
    return (c0=c0, c1=c1, W=W, a=a, w0=w0, w1=w1)
end

"""Indifference curve helper: c1 = f(c0) along utility level U."""
function c1_from_U(c0, U, β, θ)
    rhs = (U - u(c0, θ)) / β
    if abs(θ - 1.0) < 1e-8
        return exp(rhs)
    else
        val = rhs * (1 - 1/θ) + 1
        return val > 0 ? val^(1/(1 - 1/θ)) : NaN
    end
end

println("Model functions loaded.")

## 1. From Varieties to Time — Numerical Sanity Check

With discount factor $\beta$ and IES $\theta$, the canonical weights are
$$
\omega_0 = \frac{1}{1+\beta^\theta}, \qquad \omega_1 = \frac{\beta^\theta}{1+\beta^\theta},
$$
and the consumption ratio satisfies
$$
\frac{c_1^\star}{c_0^\star} = (\beta R)^\theta.
$$

In [ ]:
β = 0.96
R = 1.04
y0, y1, b0 = 1.0, 1.0, 0.0

println("θ    | c0*     c1*     c1/c0    (βR)^θ    w0      w1")
println("-----|-------------------------------------------------------")
for θ in [0.25, 0.5, 1.0, 1.5, 2.5]
    s = solve_two_period(; b0=b0, y0=y0, y1=y1, β=β, θ=θ, R=R)
    @printf("%.2f | %.4f  %.4f  %.4f   %.4f    %.3f   %.3f\n",
        θ, s.c0, s.c1, s.c1/s.c0, (β*R)^θ, s.w0, s.w1)
end

## 2. The Euler Equation and Consumption Growth

The Euler equation for CRRA utility is $c_0^{-1/\theta} = \beta R \, c_1^{-1/\theta}$, which gives
$$
\log\frac{c_1^\star}{c_0^\star} = \theta \cdot \log(\beta R).
$$
The IES is the slope of log-consumption growth against $\log(\beta R)$.

In [ ]:
β = 0.96
R_grid = range(0.90, 1.15, length=200)
θ_vec  = [0.25, 0.5, 1.0, 2.0, 4.0]
colors = [C_BLUE, C_GREEN, C_GREY, C_ORANGE, C_RED]

p1 = plot(xlabel=L"\log(\beta R)",
          ylabel=L"\log(c_1^\star / c_0^\star)",
          title="Consumption growth and the IES",
          legend=:topleft)

for (i, θ) in enumerate(θ_vec)
    growth = @. θ * log(β * R_grid)
    plot!(p1, log.(β .* R_grid), growth,
          label=L"\theta = %$θ", color=colors[i])
end

vline!(p1, [0.0], color=:black, linestyle=:dash, alpha=0.5, label=L"\beta R = 1")
hline!(p1, [0.0], color=:black, linestyle=:dot, alpha=0.3, label="")

display(p1)

## 3. Budget Set, Indifference Curves, and the Optimum

We plot the lifetime budget $c_0 + c_1/R = W$, which we can also write — using $W = y_0 + b_0 + y_1/R$ — as
$$
c_1 = R(y_0 + b_0 - c_0) + y_1.
$$
**Written this way, every budget line passes through the endowment point $(y_0 + b_0,\, y_1)$, regardless of $R$.** This is the key geometric fact we will exploit in Section 6.

In [ ]:
function plot_opt(; β=0.96, θ=1.0, R=1.04, b0=0.0, y0=1.0, y1=1.0, title_str="")
    s = solve_two_period(; β=β, θ=θ, R=R, b0=b0, y0=y0, y1=y1)
    W = s.W

    c0_grid = range(0.01, W * 0.999, length=300)
    c1_line = @. R * (y0 + b0 - c0_grid) + y1

    Ū = u(s.c0, θ) + β * u(s.c1, θ)
    c0_ic = range(0.05 * W, 1.6 * W, length=400)
    c1_ic = [c1_from_U(c, Ū, β, θ) for c in c0_ic]

    p = plot(xlabel=L"c_0", ylabel=L"c_1",
             title=title_str, legend=:topright,
             xlims=(0, 1.25*W), ylims=(0, 1.25*R*W))
    plot!(p, c0_grid, max.(c1_line, 0.0), color=C_BLUE, lw=2.5, label=L"c_0 + c_1/R = W")
    plot!(p, c0_ic, c1_ic, color=C_RED, lw=2, linestyle=:dash, label="Indiff. curve")
    scatter!(p, [y0 + b0], [y1], color=C_GREEN, markersize=6,
             markerstrokewidth=0, label="Endowment")
    scatter!(p, [s.c0], [s.c1], color=C_RED, markersize=7,
             markerstrokewidth=0, label="Optimum")
    plot!(p, [0, min(W, R*W)], [0, min(W, R*W)], color=:black,
          linestyle=:dot, alpha=0.4, label=L"c_0 = c_1")
    return p, s
end

p2, _ = plot_opt(; β=0.96, θ=1.0, R=1.04,
                 title_str=L"Budget set and optimum: $\theta = 1$, $\beta R \approx 1$")
display(p2)

## 4. The Role of the IES $\theta$

We vary $R$ and trace out how optimal $c_0$ responds for different values of $\theta$. This is the **uncompensated** (Marshallian) response — wealth is *not* held constant.

**Key point.** In the canonical static problem, the expenditure share on good $i$ is $w_i = \omega_i (p_i / P^\star)^{1-\varepsilon}$. With $\theta = \varepsilon = 1$ (Cobb-Douglas), $w_0$ is constant in prices — the intertemporal benchmark where the *compensated* substitution effect on $c_0$ vanishes.

In [ ]:
β = 0.96
y0, y1, b0 = 1.0, 1.0, 0.0
R_grid = range(0.92, 1.15, length=200)
θ_vec  = [0.25, 0.5, 1.0, 2.0, 4.0]
colors = [C_BLUE, C_GREEN, C_GREY, C_ORANGE, C_RED]

p3 = plot(xlabel=L"Gross interest rate $R$",
          ylabel=L"Optimal $c_0^\star$",
          title=L"Current consumption vs $R$ by IES ($y_0 = y_1 = 1$)",
          legend=:topright)

for (i, θ) in enumerate(θ_vec)
    c0_path = [solve_two_period(; β=β, θ=θ, R=R, b0=b0, y0=y0, y1=y1).c0 for R in R_grid]
    plot!(p3, R_grid, c0_path, label=L"\theta = %$θ", color=colors[i])
end

display(p3)

## 5. Wealth Effects from Interest Rate Changes

A crucial distinction from the static problem: an interest rate change affects **lifetime wealth** itself, because
$$
W = b_0 + y_0 + \frac{y_1}{R}.
$$
When $R$ rises, the present value of $y_1$ falls. This is a *pure* wealth effect on top of the price channel.

In [ ]:
R_grid = range(0.92, 1.20, length=200)

profiles = [
    (b0=0.0, y0=2.0, y1=0.0, color=C_BLUE,  label="All income today"),
    (b0=0.0, y0=1.0, y1=1.0, color=C_GREEN, label="Equal endowments"),
    (b0=0.0, y0=0.0, y1=2.0, color=C_RED,   label="All income tomorrow"),
]

p4 = plot(xlabel=L"Gross interest rate $R$",
          ylabel=L"Lifetime wealth $W$",
          title="Wealth effect of R depends on income timing",
          legend=:topright)

for p in profiles
    W_path = [W_life(p.b0, p.y0, p.y1, R) for R in R_grid]
    plot!(p4, R_grid, W_path, color=p.color, label=p.label)
end

vline!(p4, [1.0], color=:black, linestyle=:dot, alpha=0.4, label="")
display(p4)

## 6. Geometric Slutsky Decomposition — Budget Rotation and the Saver/Borrower Split

We now do the classical Hicksian decomposition **in the $(c_0, c_1)$ plane**. The entire saver/borrower story is already visible in one geometric fact: **when $R$ changes, the budget line rotates around the endowment point $\omega = (y_0 + b_0,\, y_1)$.** It does not pivot around a fixed wealth level, and it does not shift in parallel. Rotation around $\omega$ is the seed of every distributional result that follows.

### The decomposition

Start at the original optimum $A = (c_0^\star(R_0), c_1^\star(R_0))$, on indifference curve $U_0$. Increase the rate to $R_1 > R_0$. The new optimum is $C = (c_0^\star(R_1), c_1^\star(R_1))$, on indifference curve $U_1$.

**Substitution effect ($A \to B$).** Imagine the consumer is *compensated* to stay on $U_0$ while facing the new relative price $q_1 = 1/R_1$. The Hicksian bundle $B = (c_0^h, c_1^h)$ is the tangency between $U_0$ and a line of slope $-R_1$. Since $c_1$ became cheaper, $B$ has *more* $c_1$ and *less* $c_0$ than $A$. This move is the same qualitatively for savers and borrowers — it depends only on preferences and on the slope change.

**Income / wealth effect ($B \to C$).** The actual new budget at rate $R_1$ passes through the endowment $\omega$, not through $B$. Whether the actual budget lies *above* or *below* the compensated budget at the relevant point depends on the sign of the consumer's net position in the bond market.

### Why savers and borrowers split

Every budget line at rate $R$ passes through $\omega$ with slope $-R$. A rotation about $\omega$ that *steepens* the slope:

- moves the line **up** on the side with $c_0 < y_0$ (upper-left of $\omega$)
- moves the line **down** on the side with $c_0 > y_0$ (lower-right of $\omega$).

A **saver** consumes less than their endowment today ($c_0^\star < y_0$, i.e. $a > 0$), so $A$ lies upper-left of $\omega$. The new (steeper) budget lies *above* the old one at the saver's consumption level — the old bundle $A$ is still affordable at the new rate, *plus more*. So the saver's feasible set strictly expands in the region they care about, and they must be weakly better off.

A **borrower** consumes more than their endowment today ($c_0^\star > y_0$, i.e. $a < 0$), so $A$ lies lower-right of $\omega$. The new budget lies *below* the old at the borrower's consumption level — $A$ is no longer affordable. The feasible set strictly contracts in the relevant region, and the borrower must be worse off.

This is the cleanest geometric statement of the distributional content of a rate change: **the old optimum remains feasible for the saver but not for the borrower.**

In [ ]:
"""
Compute the Hicksian (compensated) bundle: consumption achieving utility U_target
at intertemporal price q = 1/R. Uses the Euler tangency c1/c0 = (βR)^θ
plus bisection on the utility equation.
"""
function hicksian_bundle(; β, θ, R, U_target)
    k = (β * R)^θ                             # tangency ratio c1/c0
    f(c) = u(c, θ) + β * u(k * c, θ) - U_target
    # Bisection
    lo, hi = 1e-8, 500.0
    for _ in 1:200
        mid = 0.5 * (lo + hi)
        f(mid) > 0 ? (hi = mid) : (lo = mid)
    end
    c0h = 0.5 * (lo + hi)
    return c0h, k * c0h
end

"""
Geometric Slutsky decomposition plot for a single agent.
Shows original budget at R0, new budget at R1, compensated budget,
indifference curves U0 and U1, and the three points A, B, C.
"""
function plot_slutsky_geometric(; β, θ, R0, R1, b0, y0, y1, title_str="")
    sA = solve_two_period(; β=β, θ=θ, R=R0, b0=b0, y0=y0, y1=y1)
    sC = solve_two_period(; β=β, θ=θ, R=R1, b0=b0, y0=y0, y1=y1)
    U0 = u(sA.c0, θ) + β * u(sA.c1, θ)
    U1 = u(sC.c0, θ) + β * u(sC.c1, θ)
    c0h, c1h = hicksian_bundle(; β=β, θ=θ, R=R1, U_target=U0)

    # Plot extents
    xmax = max(sA.W, sC.W, y0 + b0 + 0.2) * 1.10
    ymax = max(sA.c1, sC.c1, c1h, y1 + 0.2) * 1.40

    c0_grid = range(0.001, xmax, length=400)
    bud0 = @. R0 * (y0 + b0 - c0_grid) + y1
    bud1 = @. R1 * (y0 + b0 - c0_grid) + y1
    budh = @. c1h - R1 * (c0_grid - c0h)

    c0_ic = range(0.05 * xmax, 2.0 * xmax, length=600)
    ic0 = [c1_from_U(c, U0, β, θ) for c in c0_ic]
    ic1 = [c1_from_U(c, U1, β, θ) for c in c0_ic]

    p = plot(xlabel=L"c_0", ylabel=L"c_1",
             title=title_str, legend=:topright,
             xlims=(0, xmax), ylims=(0, ymax))

    # Budget lines
    plot!(p, c0_grid, max.(bud0, 0.0), color=C_BLUE,   lw=2,   label=L"Budget at $R_0$")
    plot!(p, c0_grid, max.(budh, 0.0), color=C_PURPLE, lw=2,   linestyle=:dash,
          label="Compensated budget")
    plot!(p, c0_grid, max.(bud1, 0.0), color=C_RED,    lw=2,   label=L"Budget at $R_1$")
    # Indifference curves
    plot!(p, c0_ic, ic0, color=C_GREY,   lw=1.4, linestyle=:dashdot, label=L"U_0")
    plot!(p, c0_ic, ic1, color=C_ORANGE, lw=1.4, linestyle=:dashdot, label=L"U_1")

    # Arrows A -> B (substitution) and B -> C (income), thicker for visibility
    plot!(p, [sA.c0, c0h], [sA.c1, c1h],
          arrow=(:closed, 0.5), color=C_PURPLE, lw=2.8, label="")
    plot!(p, [c0h, sC.c0], [c1h, sC.c1],
          arrow=(:closed, 0.5), color=:black, lw=2.8, label="")

    # Points (larger so they read as distinct)
    scatter!(p, [y0 + b0], [y1], color=C_GREEN, markersize=8,  markerstrokewidth=0, label="")
    scatter!(p, [sA.c0],   [sA.c1], color=C_BLUE,   markersize=10, markerstrokewidth=0, label="")
    scatter!(p, [c0h],     [c1h],   color=C_PURPLE, markersize=10, markerstrokewidth=0, label="")
    scatter!(p, [sC.c0],   [sC.c1], color=C_RED,    markersize=10, markerstrokewidth=0, label="")

    # Point labels — larger offsets so the letters clear the dots
    dx, dy = 0.05 * xmax, 0.05 * ymax
    annotate!(p, y0 + b0 + dx,  y1 - dy,    text(L"\omega", 13, :left,  C_GREEN))
    annotate!(p, sA.c0 + dx,    sA.c1 + dy, text(L"A",      13, :left,  C_BLUE))
    annotate!(p, c0h - dx,      c1h + dy,   text(L"B",      13, :right, C_PURPLE))
    annotate!(p, sC.c0 + dx,    sC.c1 - dy, text(L"C",      13, :left,  C_RED))

    return p, (sA=sA, sC=sC, c0h=c0h, c1h=c1h, U0=U0, U1=U1)
end

println("Geometric Slutsky functions loaded.")

### Numerical values for the decomposition

We parametrize: $\beta = 0.96$, $\theta = 2$, with a rate hike from $R_0 = 1.04$ to $R_1 = 1.40$. The saver has $(y_0, y_1) = (1.6, 0.4)$, the borrower $(y_0, y_1) = (0.4, 1.6)$.

In [ ]:
β, θ, R0, R1 = 0.96, 2.0, 1.04, 1.40
saver    = (b0=0.0, y0=1.6, y1=0.4)
borrower = (b0=0.0, y0=0.4, y1=1.6)

function print_decomp(label, prof)
    sA = solve_two_period(; β=β, θ=θ, R=R0, prof...)
    sC = solve_two_period(; β=β, θ=θ, R=R1, prof...)
    U0 = u(sA.c0, θ) + β * u(sA.c1, θ)
    U1 = u(sC.c0, θ) + β * u(sC.c1, θ)
    c0h, c1h = hicksian_bundle(; β=β, θ=θ, R=R1, U_target=U0)
    # Check: is old bundle A affordable at new rate?
    cost_A_R1 = sA.c0 + sA.c1/R1
    W1 = W_life(prof.b0, prof.y0, prof.y1, R1)
    slack = W1 - cost_A_R1

    println("--- $(label) ---")
    @printf("  A (old opt):      c0 = %.4f,  c1 = %.4f,  U = %+.5f,  a = %+.4f\n",
            sA.c0, sA.c1, U0, sA.a)
    @printf("  B (Hicksian):     c0 = %.4f,  c1 = %.4f,  (on U0)\n", c0h, c1h)
    @printf("  C (new opt):      c0 = %.4f,  c1 = %.4f,  U = %+.5f\n",
            sC.c0, sC.c1, U1)
    @printf("  Substitution A→B: Δc0 = %+.4f,  Δc1 = %+.4f\n", c0h-sA.c0, c1h-sA.c1)
    @printf("  Income       B→C: Δc0 = %+.4f,  Δc1 = %+.4f\n", sC.c0-c0h, sC.c1-c1h)
    @printf("  Welfare ΔU:       %+.5f   (%s)\n", U1-U0,
            U1 > U0 ? "better off" : "worse off")
    @printf("  W_1 − cost(A at R_1) = %+.4f   (%s)\n", slack,
            slack > 0 ? "A still affordable" : "A no longer affordable")
    println()
end

print_decomp("Saver",    saver)
print_decomp("Borrower", borrower)

Notice the last line in each block: for the saver, $W_1 - \text{cost}(A\text{ at }R_1) > 0$, meaning the old optimum $A$ is still affordable under the new budget — with slack. For the borrower, the slack is negative — $A$ is no longer feasible. Revealed preference alone then gives $U_1 > U_0$ for the saver and $U_1 < U_0$ for the borrower. This is before we even look at indifference curves.

In [ ]:
# Side-by-side: saver (left), borrower (right)
pS, _ = plot_slutsky_geometric(; β=β, θ=θ, R0=R0, R1=R1,
           b0=0.0, y0=1.6, y1=0.4,
           title_str=L"Saver: $y_0=1.6,\ y_1=0.4$")
pB, _ = plot_slutsky_geometric(; β=β, θ=θ, R0=R0, R1=R1,
           b0=0.0, y0=0.4, y1=1.6,
           title_str=L"Borrower: $y_0=0.4,\ y_1=1.6$")

pSB = plot(pS, pB, layout=(1,2), size=(1300, 600))
display(pSB)

**Reading the figure.**

Follow the three points in each panel: $A$ (blue) is the old optimum at $R_0$; $B$ (purple) is the Hicksian bundle on the *old* indifference curve $U_0$, tangent to a line of slope $-R_1$; $C$ (red) is the new optimum at $R_1$.

The **purple arrow** ($A \to B$) is the pure substitution effect. It points up and to the left in both panels — the direction of substitution is the same for savers and borrowers. Magnitudes differ slightly because the two agents sit on different indifference curves, but the elasticity governing this arrow is the same: it depends only on $\theta$ and the share $w_1$.

The **black arrow** ($B \to C$) is the income/wealth effect.
- *Saver*: $B \to C$ moves **up and to the right** — the new budget lies outside the compensated budget at $B$, so the consumer reaches a higher indifference curve ($U_1 > U_0$, orange dash-dot line above the gray one).
- *Borrower*: $B \to C$ moves **down and to the left** — the new budget lies inside the compensated budget at $B$, so the consumer reaches a lower indifference curve ($U_1 < U_0$, orange dash-dot line below the gray one).

Geometrically, the saver/borrower split is the sign of the gap between the red and purple lines at the point $B$:

- saver: $C$ lies on a budget line *above* the compensated one → wealth effect is expansionary;
- borrower: $C$ lies on a budget line *below* the compensated one → wealth effect is contractionary.

This is why a rate hike is distributionally a transfer from borrowers to savers — visible from the geometry of the rotation alone.

## 7. The Algebraic Hicksian Slutsky Decomposition

The same decomposition can be written algebraically. Differentiating the Euler equation and the budget constraint gives

$$
\underbrace{\frac{d \log c_0}{d \log R}}_{\text{total}}
\;=\; \underbrace{(1 - \theta) \, w_1}_{\text{compensated substitution}}
\;\;-\;\; \underbrace{\frac{y_1}{R W}}_{\text{PV-wealth channel}}.
$$

The substitution term isolates the response **holding lifetime wealth $W$ constant** — it is the algebraic counterpart of the arrow $A \to B$ in Section 6 converted to an elasticity. It depends only on $\theta$ and $w_1$. The wealth term $-y_1/(RW)$ is always negative when $y_1 > 0$: higher $R$ lowers the PV of future income, a pure Marshallian wealth effect.

At the **Cobb-Douglas benchmark $\theta = 1$**, the substitution term vanishes and the entire response is the wealth channel.

In [ ]:
"""
Hicksian Slutsky decomposition of d log c0 / d log R.
Returns (total, substitution, wealth, state).
"""
function slutsky_hicksian(; β, θ, R, b0, y0, y1)
    s  = solve_two_period(; β=β, θ=θ, R=R, b0=b0, y0=y0, y1=y1)
    sub = (1 - θ) * s.w1
    wealth = -y1 / (R * s.W)
    return sub + wealth, sub, wealth, s
end

# Sanity check: compare analytical formula to a finite-difference derivative
β, θ = 0.96, 2.0
println("agent      sub       wealth    total     numeric")
println("-----------------------------------------------------")
for (lbl, y0, y1) in [("Saver",    1.6, 0.4),
                      ("Borrower", 0.4, 1.6),
                      ("Equal",    1.0, 1.0)]
    R = 1.04; h = 1e-4
    t, sub, w, _ = slutsky_hicksian(; β=β, θ=θ, R=R, b0=0, y0=y0, y1=y1)
    s_p = solve_two_period(; β=β, θ=θ, R=R+h, b0=0, y0=y0, y1=y1)
    s_m = solve_two_period(; β=β, θ=θ, R=R-h, b0=0, y0=y0, y1=y1)
    num = (log(s_p.c0) - log(s_m.c0)) / (log(R+h) - log(R-h))
    @printf("%-10s %+.4f   %+.4f   %+.4f   %+.4f\n", lbl, sub, w, t, num)
end

In [ ]:
# Figure: Hicksian decomposition for saver vs borrower across R
β, θ = 0.96, 2.0
R_grid = range(0.95, 1.12, length=180)

function decomp_path_hick(grid, prof)
    T, S, W = zeros(length(grid)), zeros(length(grid)), zeros(length(grid))
    for (i, R) in enumerate(grid)
        t, s, w, _ = slutsky_hicksian(; β=β, θ=θ, R=R, prof...)
        T[i]=t; S[i]=s; W[i]=w
    end
    return T, S, W
end

T_s, S_s, W_s = decomp_path_hick(R_grid, saver)
T_b, S_b, W_b = decomp_path_hick(R_grid, borrower)

pL = plot(R_grid, S_s, color=C_BLUE,  label="Substitution",
          xlabel=L"R", ylabel=L"d\log c_0 / d\log R",
          title=L"Saver ($y_0=1.6,\,y_1=0.4$)", legend=:bottomleft)
plot!(pL, R_grid, W_s, color=C_GREEN, label="Wealth (PV)")
plot!(pL, R_grid, T_s, color=C_RED,   label="Total", lw=2.5)
hline!(pL, [0.0], color=:black, linestyle=:dot, alpha=0.4, label="")

pR = plot(R_grid, S_b, color=C_BLUE,  label="Substitution",
          xlabel=L"R", ylabel=L"d\log c_0 / d\log R",
          title=L"Borrower ($y_0=0.4,\,y_1=1.6$)", legend=:bottomleft)
plot!(pR, R_grid, W_b, color=C_GREEN, label="Wealth (PV)")
plot!(pR, R_grid, T_b, color=C_RED,   label="Total", lw=2.5)
hline!(pR, [0.0], color=:black, linestyle=:dot, alpha=0.4, label="")

p5 = plot(pL, pR, layout=(1,2), size=(1100, 450))
display(p5)

**Reading the figure.** The Hicksian (compensated) substitution effect is *identical* for saver and borrower — it depends only on $\theta$ and $w_1$. What differs across agents is the **wealth channel**: its magnitude is governed by $y_1/(RW)$, the PV-weight of future income. The borrower, with $y_1 = 1.6$, loses far more present-value wealth when $R$ rises.

## 8. Savers vs. Borrowers: The Price + Savings Decomposition

The Hicksian decomposition above is the textbook version. An **alternative decomposition** makes the saver/borrower distinction more directly visible by isolating the agent's *savings position*:

$$
\frac{d \log c_0}{d \log R}
\;=\; \underbrace{-\theta \, w_1}_{\text{price / smoothing channel}}
\;+\; \underbrace{\frac{a}{W}}_{\text{savings channel}}.
$$

The price channel is always negative: higher $R$ makes $c_1$ cheaper, so the consumer tilts consumption toward tomorrow. Its magnitude scales with $\theta$. The savings channel carries the **sign of savings**: positive for savers ($a > 0$), negative for borrowers ($a < 0$).

**Equivalence.** Using $a/W = w_1 - y_1/(RW)$ (from $a = (c_1-y_1)/R$ and $w_1 = c_1/(RW)$), one can check directly that $(1-\theta)w_1 - y_1/(RW) = -\theta w_1 + a/W$. The two decompositions are algebraically the same total, split in two different ways.

In [ ]:
"""Alternative decomposition: price (smoothing) + savings-position channel."""
function slutsky_savings(; β, θ, R, b0, y0, y1)
    s  = solve_two_period(; β=β, θ=θ, R=R, b0=b0, y0=y0, y1=y1)
    price   = -θ * s.w1
    savings = s.a / s.W
    return price + savings, price, savings, s
end

β, θ = 0.96, 2.0
R_grid = range(0.95, 1.12, length=180)

function decomp_path_sav(grid, prof)
    T, P, S = zeros(length(grid)), zeros(length(grid)), zeros(length(grid))
    for (i, R) in enumerate(grid)
        t, p, s, _ = slutsky_savings(; β=β, θ=θ, R=R, prof...)
        T[i]=t; P[i]=p; S[i]=s
    end
    return T, P, S
end

T_s, P_s, Sv_s = decomp_path_sav(R_grid, saver)
T_b, P_b, Sv_b = decomp_path_sav(R_grid, borrower)

qL = plot(R_grid, P_s,  color=C_BLUE,  label="Price",
          xlabel=L"R", ylabel=L"d\log c_0 / d\log R",
          title=L"Saver: $a/W > 0$", legend=:bottomleft)
plot!(qL, R_grid, Sv_s, color=C_GREEN, label="Savings")
plot!(qL, R_grid, T_s,  color=C_RED,   label="Total", lw=2.5)
hline!(qL, [0.0], color=:black, linestyle=:dot, alpha=0.4, label="")

qR = plot(R_grid, P_b,  color=C_BLUE,  label="Price",
          xlabel=L"R", ylabel=L"d\log c_0 / d\log R",
          title=L"Borrower: $a/W < 0$", legend=:bottomleft)
plot!(qR, R_grid, Sv_b, color=C_GREEN, label="Savings")
plot!(qR, R_grid, T_b,  color=C_RED,   label="Total", lw=2.5)
hline!(qR, [0.0], color=:black, linestyle=:dot, alpha=0.4, label="")

p6 = plot(qL, qR, layout=(1,2), size=(1100, 450))
display(p6)

**Reading the figure.** The **price channel** (blue) is the same negative line for both agents — purely the consumption-smoothing pull. The **savings channel** (green) carries the sign of $a$: positive for the saver, negative for the borrower. It partially offsets the price channel for the saver and reinforces it for the borrower.

## 9. Welfare: Who Gains, Who Loses?

The consumption response of $c_0$ tells only part of the distributional story. The more direct question is: *does a rate change make the agent better or worse off?* We plot lifetime utility $V = u(c_0^\star) + \beta \, u(c_1^\star)$ for saver and borrower as $R$ varies.

In [ ]:
β, θ = 0.96, 2.0
R_grid = range(0.95, 1.15, length=200)

function V_path(prof)
    V = zeros(length(R_grid))
    for (i, R) in enumerate(R_grid)
        s = solve_two_period(; β=β, θ=θ, R=R, prof...)
        V[i] = u(s.c0, θ) + β * u(s.c1, θ)
    end
    return V
end

V_s = V_path(saver)
V_b = V_path(borrower)

i_ref = argmin(abs.(R_grid .- 1.04))
V_s_n = V_s .- V_s[i_ref]
V_b_n = V_b .- V_b[i_ref]

p8 = plot(R_grid, V_s_n, color=C_BLUE, label="Saver",
          xlabel=L"R", ylabel=L"V(R) - V(R_0)",
          title="Welfare effect of R: savers vs. borrowers",
          legend=:topright, lw=2.5)
plot!(p8, R_grid, V_b_n, color=C_RED, label="Borrower", lw=2.5)
vline!(p8, [1.04], color=:black, linestyle=:dot, alpha=0.4, label="")
hline!(p8, [0.0],  color=:black, linestyle=:dot, alpha=0.4, label="")

display(p8)

**Reading the figure.** Around $R_0 = 1.04$, the saver's welfare is *monotonically increasing* in $R$ while the borrower's is *monotonically decreasing*. By the envelope theorem, $dV/dR = \beta \, u'(c_1^\star) \cdot a$ at the optimum — positive for a saver, negative for a borrower. The welfare sign is determined entirely by the savings position, independent of $\theta$.

## 10. Exercises

**Exercise 1 — Verify the Euler equation numerically.** For $\beta = 0.96$, $\theta = 2$, $R = 1.05$, and endowments $(y_0, y_1) = (1, 1)$, solve the problem and check that $c_0^{-1/\theta} = \beta R \, c_1^{-1/\theta}$ holds to machine precision. Repeat for $\theta = 0.5$ and $\theta = 1.0$.

**Exercise 2 — The Cobb-Douglas benchmark.** Show numerically that for $\theta = 1$, the expenditure share $w_0$ is independent of $R$. Does this mean $c_0^\star$ itself is independent of $R$? Why not? Use the geometric decomposition in Section 6 to explain.

**Exercise 3 — Rate decrease.** Redo the geometric decomposition in Section 6 for a rate *decrease* from $R_0 = 1.10$ to $R_1 = 1.00$. Verify that the arrows reverse direction and that the saver is now worse off while the borrower is better off.

**Exercise 4 — Revealed preference.** Using only the fact that the budget line rotates around the endowment $\omega$, argue without calculus that a saver cannot be worse off after a rate increase and a borrower cannot be better off. (Hint: show that the old optimum $A$ remains affordable for the saver but not for the borrower.)

**Exercise 5 — Capital taxation, compensated vs. uncompensated.** Modify the model to include a tax $\tau$ on capital income so the after-tax return is $R(1-\tau)$. Compare two policies: (a) no rebate, (b) lump-sum rebate of the tax revenue. Plot $c_0^\star$ as a function of $\tau$ for both. Which isolates the Hicksian (compensated) response? Which is the Marshallian (uncompensated) response?

**Exercise 6 — Equivalence of decompositions.** Verify numerically that the Hicksian decomposition and the price + savings decomposition give identical totals for any $(\theta, R, y_0, y_1)$. Show algebraically that $(1-\theta)w_1 - y_1/(RW) = -\theta w_1 + a/W$ using the identities $a = (c_1 - y_1)/R$ and $w_1 = c_1/(RW)$.

---

**End of notebook.**